# Sector news-drift model — full implementation

Predicts each synthetic stock's market-adjusted return from the close of bar 49 to the close of bar 59, using only bars and headlines 0–49. Score = annualised Sharpe of `pred × return`.

**In-sample 5-fold CV Sharpe: 3.546 / 3.527 / 3.530** (three shuffles).

| Step | What | CV Sharpe after step |
|---|---|---|
| 1 | Keep only headlines about the stock's own sector | — |
| 2–3 | Per-template drift, fitted jointly; news signal = drift still to come after bar 49, ÷ vol | 3.38 |
| 4 | + residual momentum (unexplained recent moves continue) | 3.45 |
| 5 | + Sharpe-optimal sizing E[r]/E[r²] | **3.55** |

Data: put the five parquet files (`ins_bars`, `ins_headlines`, `ins_returns`, `oos_bars`, `oos_headlines`) in `DATA` below. This is the same code as `submissions/sector_model.py`, and the last cell writes the identical submission CSV.

In [1]:
import re, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import lsqr
from sklearn.cluster import SpectralClustering
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

DATA = next((p for p in (Path("../data"), Path("data"), Path(".."), Path(".")) if (p / "ins_bars.parquet").exists()), Path("data"))

HALF = 49                       # trade at the close of this bar
HORIZON = 10                    # hold until the close of bar 59
DRIFT_LEN = 10                  # a headline drifts bars b .. b+DRIFT_LEN
N_SECTORS = 5
SUFFIX = {"Group", "Co"}        # third word that is still part of a company name
LAM = 200.0                     # template ridge strength (scaled by median vol^2)
HALF_LIVES = (1, 2, 3, 5, 10, 25, 1000)
ALPHA = 50.0                    # calibration ridge strength
print("data dir:", DATA.resolve())

data dir: /Users/chfm/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Projects/AlgoArena/data


## 1. Headlines: templates, timing and sectors

**Templates.** Headlines come from about 300 phrase templates. Remove the company name and normalise amounts (`$500M` → `$X`, `7%` → `N%`).

**Timing.** A headline at bar *b* pushes the price for 10 bars, then stops. The weights are 0.5 on bar *b*, 1 on each of the next 9 bars, and 0.5 on bar *b*+10 (measured from the data). The model trades from bar 49 to bar 59, so only the part of the push that falls in that window counts. A headline at bar 47 has 7.5 bars left; one at bar 39 or earlier has none.

**Sectors.** The 80 companies form 5 sectors of 16. A session's stock is in the sector with the most distinct companies among its headlines; headlines about other sectors have **zero** effect on its price.

In [2]:
def drift_weight(k):
    """Weight of the k-th bar after a headline (k = 0 is the headline's own bar)."""
    return 0.5 if k in (0, DRIFT_LEN) else 1.0


def remaining_drift(b):
    """Drift bars of a headline at bar b that fall in the holding window (HALF, HALF + HORIZON]."""
    b = np.asarray(b)
    return sum(drift_weight(k) * ((b + k > HALF) & (b + k <= HALF + HORIZON)) for k in range(DRIFT_LEN + 1))


def parse_headlines(hl):
    hl = hl.copy()
    words = hl.headline.str.split()
    hl["company"] = [" ".join(w[:3]) if w[2] in SUFFIX else " ".join(w[:2]) for w in words]
    rest = [" ".join(w[3:]) if w[2] in SUFFIX else " ".join(w[2:]) for w in words]
    hl["tmpl"] = [re.sub(r"\d+(\.\d+)?%", "N%", re.sub(r"\$[\d.]+[MBK]?", "$X", r)) for r in rest]
    return hl


def company_sectors(hl):
    """Cluster companies by how often they appear in the same session."""
    comp = np.sort(hl.company.unique())
    cix = {c: i for i, c in enumerate(comp)}
    M = np.zeros((len(comp), len(comp)))
    for cs in hl.groupby("session").company.apply(lambda s: sorted(set(s))):
        for a in cs:
            for b in cs:
                if a != b:
                    M[cix[a], cix[b]] += 1
    labels = SpectralClustering(N_SECTORS, affinity="precomputed", random_state=0).fit_predict(M + 1e-3)
    return dict(zip(comp, labels))

print("remaining drift for headlines at bars 38..49:", dict(zip(range(38, 50), remaining_drift(np.arange(38, 50)))))

remaining drift for headlines at bars 38..49: {38: np.float64(0.0), 39: np.float64(0.0), 40: np.float64(0.5), 41: np.float64(1.5), 42: np.float64(2.5), 43: np.float64(3.5), 44: np.float64(4.5), 45: np.float64(5.5), 46: np.float64(6.5), 47: np.float64(7.5), 48: np.float64(8.5), 49: np.float64(9.5)}


In [3]:
raw_ins = parse_headlines(pd.read_parquet(DATA / "ins_headlines.parquet"))
sectors = company_sectors(raw_ins)
for g in range(N_SECTORS):
    members = sorted(c for c, s in sectors.items() if s == g)
    print(f"sector {g} ({len(members)} companies):", ", ".join(members[:5]), "...")

sector 0 (16 companies): Arnik Commerce, Arnos Marketplace, Crevol Retail Group, Frelex Outlets, Halvav Brands ...
sector 1 (16 companies): Creven Securities, Ervan Capital, Halven Investments, Halvix Holdings, Holtum Asset Group ...
sector 2 (16 companies): Jorvis Fuels, Kelvik Power, Kelvos Resources, Nerven Grid, Nolvol Resources ...
sector 3 (16 companies): Brevep Systems, Brevon Microchips, Crevex Labs, Dralol Computing, Frelol Software ...
sector 4 (16 companies): Calvis Sciences, Calvos Genomics, Frelis Genomics, Grevel Diagnostics, Grevon Biotech ...


## 2. Load each split: prices, main-sector headlines, volatility

The main sector is chosen from headlines in bars 0–49 only, with ties broken by headline count. It matches the sector implied by all 100 bars in 99.4% of in-sample sessions. Volatility is the Garman–Klass estimate from bars 0–49 open/high/low/close.

In [4]:
def load_split(kind, sectors):
    """Prices (log), main-sector headlines and Garman-Klass vol for one split ('ins' or 'oos')."""
    bars = pd.read_parquet(DATA / f"{kind}_bars.parquet")
    hl = parse_headlines(pd.read_parquet(DATA / f"{kind}_headlines.parquet"))
    S = np.sort(bars.session.unique())
    P = {c: np.log(bars.pivot(index="session", columns="bar_ix", values=c).reindex(S).to_numpy())
         for c in ("open", "high", "low", "close")}

    hl["grp"] = hl.company.map(sectors)
    if hl.grp.isna().any():
        raise ValueError("headline mentions a company not seen in-sample")
    h49 = hl[hl.bar_ix <= HALF]                                    # sector choice uses bars <= 49 only
    dist = h49.groupby(["session", "grp"]).company.nunique().unstack(fill_value=0).reindex(S, fill_value=0).to_numpy()
    cnt = h49.groupby(["session", "grp"]).size().unstack(fill_value=0).reindex(S, fill_value=0).to_numpy()
    main = np.argmax(dist * 1000 + cnt, 1)
    hl["i"] = hl.session.map({s: i for i, s in enumerate(S)})
    hl = hl[hl.grp.to_numpy() == main[hl.i.to_numpy()]].copy()

    O, H, L, C = (P[c][:, :HALF + 1] for c in ("open", "high", "low", "close"))
    gk = np.sqrt(np.clip((0.5 * (H - L) ** 2 - (2 * np.log(2) - 1) * (C - O) ** 2).mean(1), 1e-10, None))
    return dict(S=S, C=P["close"], hl=hl, gk=gk)


ins, oos = load_split("ins", sectors), load_split("oos", sectors)
y = pd.read_parquet(DATA / "ins_returns.parquet").set_index("session").returns.reindex(ins["S"]).to_numpy()
print(f"in-sample sessions {len(ins['S'])}, OOS sessions {len(oos['S'])}")
print(f"main-sector headlines kept: ins {len(ins['hl'])} / {len(raw_ins)}")

in-sample sessions 10000, OOS sessions 10000
main-sector headlines kept: ins 137652 / 174327


## 3. The model

**Template drift (steps 2–3).** Fit one coefficient per template by sparse ridge regression on every bar return. Each return is divided by its stock's volatility, because the push scales with volatility. The regressors are the pushes of all headlines active on that bar, so overlapping headlines are fitted together. The fit uses all bars of the labelled sessions plus bars 0–49 of the sessions being predicted; those are inputs with no labels. **News signal** = Σ coefficient × push still to come after bar 49.

**Residual momentum (step 4).** Take bar returns 1–49 minus the push explained by headlines. Sum them with exponential decay, at half-lives of 1–1000 bars. Recent unexplained moves continue; the whole-path move reverses slightly.

**Calibration and sizing (step 5).** A ridge regression of return ÷ vol on [news signal, 7 residual sums] gives *m* = E[r/vol | x]. The position is *m* / (vol · (Var(r/vol) + *m*²)) = E[r]/E[r²], which maximises Sharpe. It shrinks high-volatility bets and damps the very largest predictions.

In [5]:
class SectorDriftModel:
    def __init__(self, ins, oos):
        self.tm = np.sort(pd.concat([ins["hl"].tmpl, oos["hl"].tmpl]).unique())
        self.tix = {t: k for k, t in enumerate(self.tm)}
        self.lam = LAM / np.median(ins["gk"]) ** 2

    # template drift --------------------------------------------------------
    def _design(self, d, idx, maxbar):
        """Rows = (session, bar return 1..maxbar); vol-normalised returns and drift exposures."""
        C = d["C"][idx][:, :maxbar + 1]
        R = np.diff(C, axis=1)
        R = R - R.mean(0)                                          # market-adjust per bar
        nb = R.shape[1]
        h = d["hl"][d["hl"].i.isin(idx)]
        ri = h.i.map({i: r for r, i in enumerate(idx)}).to_numpy()
        b = h.bar_ix.to_numpy()
        ci = h.tmpl.map(self.tix).to_numpy()
        rows, cols, vals = [], [], []
        for k in range(DRIFT_LEN + 1):
            j = b + k - 1                                          # bar b+k's return is column b+k-1
            ok = (j >= 0) & (j < nb)
            rows.append(ri[ok] * nb + j[ok]); cols.append(ci[ok]); vals.append(np.full(ok.sum(), drift_weight(k)))
        X = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                          shape=(len(idx) * nb, len(self.tm)))
        X = sp.diags(1 / np.repeat(d["gk"][idx], nb)) @ X
        R = R / d["gk"][idx][:, None]
        return X, R.ravel()

    def fit_templates(self, parts):
        """parts: list of (dataset, row indices, last bar usable)."""
        Xs, Rs = zip(*[self._design(d, idx, maxbar) for d, idx, maxbar in parts])
        self.coef = lsqr(sp.vstack(Xs).tocsr(), np.concatenate(Rs), damp=np.sqrt(self.lam))[0]
        return self

    # features ----------------------------------------------------------------
    def news_signal(self, d, idx):
        h = d["hl"][d["hl"].i.isin(idx) & (d["hl"].bar_ix <= HALF)]
        e = h.tmpl.map(dict(zip(self.tm, self.coef))).fillna(0) * remaining_drift(h.bar_ix.to_numpy())
        return e.groupby(h.i).sum().reindex(idx).fillna(0).to_numpy()

    def residuals(self, d, idx):
        """Bar returns 1..49 (vol units) minus the drift explained by headlines at bars <= 49."""
        cm = dict(zip(self.tm, self.coef))
        h = d["hl"][d["hl"].i.isin(idx) & (d["hl"].bar_ix <= HALF)]
        c = h.tmpl.map(cm).fillna(0).to_numpy()
        b = h.bar_ix.to_numpy()
        i = h.i.map({v: k for k, v in enumerate(idx)}).to_numpy()
        R = np.diff(d["C"][idx][:, :HALF + 1], axis=1)
        R = (R - R.mean(0)) / d["gk"][idx][:, None]
        E = np.zeros_like(R)
        for k in range(DRIFT_LEN + 1):
            j = b + k - 1
            ok = (j >= 0) & (j < HALF)
            np.add.at(E, (i[ok], j[ok]), drift_weight(k) * c[ok])
        return R - E

    def features(self, d, idx):
        Z = self.residuals(d, idx)
        age = np.arange(HALF)[::-1]                                # bars before bar 49
        return np.column_stack([self.news_signal(d, idx)] + [(Z * np.exp(-age / h)).sum(1) for h in HALF_LIVES])

    # calibration + sizing ----------------------------------------------------
    def fit_calibration(self, ins, tr_idx, y_tr):
        Xa = self.features(ins, tr_idx)
        ya = y_tr / ins["gk"][tr_idx]
        self.mu, self.sd = Xa.mean(0), Xa.std(0) + 1e-12
        self.ridge = Ridge(alpha=ALPHA).fit((Xa - self.mu) / self.sd, ya)
        self.var_y = ya.var()
        return self

    def predict(self, d, idx):
        m = self.ridge.predict((self.features(d, idx) - self.mu) / self.sd) - self.ridge.intercept_
        return m / (d["gk"][idx] * (self.var_y + m ** 2))          # E[r] / E[r^2]


def train_predict(tr_idx, te, te_idx, label_free_parts):
    """Fit on labelled in-sample rows tr_idx, predict rows te_idx of dataset te."""
    model = SectorDriftModel(ins, oos)
    model.fit_templates([(ins, tr_idx, 99)] + label_free_parts)
    model.fit_calibration(ins, tr_idx, y[tr_idx])
    return model.predict(te, te_idx)


def sharpe(pred, ret):
    pnl = pred * ret
    return float(pnl.mean() / pnl.std() * np.sqrt(252))

## 4. Cross-validation (in-sample)

Five folds over the 10,000 labelled sessions. In each fold the held-out sessions' bars 0–49 and the OOS bars 0–49 are used only as label-free price data, exactly as at submission time. One shuffle takes about 1–2 minutes; set `SEEDS = (0, 1, 2)` to reproduce all three numbers.

In [6]:
SEEDS = (0,)
N, allo = len(ins["S"]), np.arange(len(oos["S"]))
for seed in SEEDS:
    folds = list(KFold(5, shuffle=True, random_state=seed).split(np.arange(N)))
    p = np.zeros(N)
    for a, b in folds:
        p[b] = train_predict(a, ins, b, [(ins, b, HALF), (oos, allo, HALF)])
    print(f"seed {seed}: 5-fold Sharpe {sharpe(p, y):.3f}  folds {[round(sharpe(p[b], y[b]), 2) for _, b in folds]}")

seed 0: 5-fold Sharpe 3.546  folds [3.42, 3.47, 3.87, 3.31, 3.67]


## 5. Fit on all in-sample data and write the submission

The model is fitted on all 10,000 labelled sessions, plus the OOS bars 0–49 as label-free price data. The CSV has columns `headline_id,pred`, with one row per OOS session.

In [7]:
pred = train_predict(np.arange(N), oos, allo, [(oos, allo, HALF)])
out = Path("sector_resid_sized_nb.csv")
pd.DataFrame({"headline_id": oos["S"].astype(np.int64), "pred": pred}).to_csv(out, index=False, lineterminator="\n")

sub = pd.read_csv(out)
expected = set(pd.read_parquet(DATA / "oos_bars.parquet").session.unique())
assert list(sub.columns) == ["headline_id", "pred"], "wrong columns"
assert len(sub) == 10000 and sub.headline_id.is_unique, "wrong row count or duplicate ids"
assert set(sub.headline_id) == expected, "ids do not match OOS sessions"
assert np.isfinite(sub.pred).all(), "non-finite predictions"
print(f"wrote {out.resolve()}  rows={len(sub)}  sha256={hashlib.sha256(out.read_bytes()).hexdigest()[:16]}")
sub.head()

wrote /Users/chfm/Library/CloudStorage/OneDrive-UniversityofEdinburgh/Projects/AlgoArena/submissions/sector_resid_sized_nb.csv  rows=10000  sha256=e807bdf5ac27937a


,headline_id,pred
0,11000,-4.102418
1,11001,19.153373
2,11002,19.786063
3,11003,-44.503172
4,11004,-40.151928
